# Lesson 01 — Types of Noise

## Why This Lesson
Real-world images are always noisy. Different noise sources produce different patterns.
Knowing your noise type determines your filter choice. Wrong filter = worse result.

## The Three Main Types
| Noise Type | Cause | Appearance |
|---|---|---|
| Gaussian | Sensor heat, electronics | Random smooth grain everywhere |
| Salt & Pepper | Transmission errors, dead pixels | Random white and black dots |
| Speckle | Ultrasound, radar, coherent light | Multiplicative granular pattern |

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

img  = cv2.imread('sample.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)

# Gaussian noise — additive, normally distributed
def add_gaussian(img, sigma=25):
    noise  = np.random.normal(0, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)

# Salt and pepper — random white/black dots
def add_salt_pepper(img, density=0.04):
    noisy  = img.copy()
    n      = int(density * img.size)
    coords = [np.random.randint(0, i-1, n) for i in img.shape[:2]]
    noisy[coords[0], coords[1]] = 255
    coords = [np.random.randint(0, i-1, n) for i in img.shape[:2]]
    noisy[coords[0], coords[1]] = 0
    return noisy

# Speckle — multiplicative noise
def add_speckle(img, sigma=0.15):
    noise = np.random.normal(1, sigma, img.shape).astype(np.float32)
    return np.clip(img.astype(np.float32) * noise, 0, 255).astype(np.uint8)

g = gray.astype(np.uint8)
gaussian_noisy    = add_gaussian(g, 30)
saltpepper_noisy  = add_salt_pepper(g, 0.04)
speckle_noisy     = add_speckle(g, 0.2)

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, im, t in zip(axes,
    [g, gaussian_noisy, saltpepper_noisy, speckle_noisy],
    ['Clean', 'Gaussian noise', 'Salt & Pepper', 'Speckle']):
    ax.imshow(im, cmap='gray'); ax.set_title(t); ax.axis('off')
plt.show()

# Histograms of each noise type
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, im, t in zip(axes,
    [gaussian_noisy, saltpepper_noisy, speckle_noisy],
    ['Gaussian — smooth histogram spread', 'Salt&Pepper — spikes at 0 and 255', 'Speckle — asymmetric spread']):
    ax.hist(im.flatten(), 256, [0,256], color='gray', edgecolor='none')
    ax.set_title(t, fontsize=10)
plt.tight_layout(); plt.show()

## Key Takeaway
Gaussian noise → Gaussian blur or NLM denoising.
Salt & Pepper → Median blur (only correct choice).
Speckle → Bilateral filter or NLM.